# List COGs for collection `2024_bci`

This notebook queries the Kanopia STAC API and lists all COG assets matching the `2024_bci` collection and the `*bciwhole*rgb.cog.tif` pattern. It is based on the structure of `workshop_kanopia_stac_cog_python.ipynb`.

In [2]:
STAC_API_URL = "https://kanopia.org/stac-fastapi-pgstac/api/v1/pgstac/"
COLLECTION_ID = "2024_bci"

print("STAC endpoint:", STAC_API_URL)
print("Collection:", COLLECTION_ID)

STAC endpoint: https://kanopia.org/stac-fastapi-pgstac/api/v1/pgstac/
Collection: 2024_bci


In [3]:
# Optional dependencies: pystac-client
# Install in your environment if missing:
#   python -m pip install pystac-client

from pystac_client import Client
import re

pattern = re.compile(r"(\d{8})_bciwhole_.*rgb\.cog\.tif")

client = Client.open(STAC_API_URL)
search = client.search(collections=[COLLECTION_ID], max_items=500)

# Support both older and new .get_all_items behaviour
items = search.get_all_items()
if hasattr(items, 'features'):
    items = items.features

cog_list = []
for item in items:
    assets = item.assets if hasattr(item, 'assets') else item.get('assets', {})
    item_id = getattr(item, 'id', None) or item.get('id', None)

    for key, asset in (assets.items() if isinstance(assets, dict) else []):
        if not asset:
            continue
        href = asset.href if hasattr(asset, 'href') else asset.get('href', '')
        if not href:
            continue

        if pattern.search(href):
            date_int = int(pattern.search(href).group(1))
            cog_list.append({
                'item_id': item_id,
                'asset_key': key,
                'href': href,
                'date': date_int,
            })

# Sort by date
cog_list.sort(key=lambda x: x['date'])

print(f"Found {len(cog_list)} matching COG assets for collection {COLLECTION_ID}")
for i, c in enumerate(cog_list, start=1):
    print(f"{i:03d}: {c['date']} | {c['item_id']} | {c['asset_key']} | {c['href']}")

ModuleNotFoundError: No module named 'pystac_client'

## Notes
- If no entries are found, ensure the `COLLECTION_ID` exists and the API endpoint is available.
- You can customize `max_items`, regex pattern, or add grid mapping to focus on date areas.